# Load deps

In [ ]:
!pip install -q sacrebleu evaluate

In [ ]:
import numpy as np
import os, torch, evaluate
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments
from transformers import Seq2SeqTrainer

# HF Login

In [ ]:
secret_label = "HF_TOKEN"
os.environ[secret_label] = UserSecretsClient().get_secret(secret_label)
!hf auth whoami

# Config

In [ ]:
hf_user_name = "amin-oj"
lang1="en"
lang2="fr"
model_checkpoint = f"Helsinki-NLP/opus-mt-{lang1}-{lang2}"

dataset_dict = dict(
    path = "kde4",
    revision = "refs/convert/parquet",
    data_dir = f"{lang1}-{lang2}"
)
push_to_hub=True
train_bs = 32
eval_bs = 64
num_train_epochs = 2
task = "translation"
max_length = 128
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-{task}-{dataset_dict['path']}"
print(ckpt_name)

# Load data

In [ ]:
raw_datasets = load_dataset(**dataset_dict)

In [ ]:
print(raw_datasets)
print("===========================")
split_datasets = raw_datasets["train"].train_test_split(train_size=0.9, seed=20)
split_datasets["validation"] = split_datasets.pop("test")
print(split_datasets)
print("===========================")
sample = split_datasets["train"][1]
print(sample["translation"])
print("===========================")

# Load and test pretrained checkpoint

- One particularity of this dataset full of technical computer science terms is that they are all fully translated in French.
- However, French engineers leave most computer science-specific words in English when they talk.
- Here, for instance, the word “threads” might well appear in a French sentence, especially in a technical conversation; but in this dataset it has been translated into the more correct “fils de discussion.”
- The pretrained model we use, which has been pretrained on a larger corpus of French and English sentences, takes the easier option of leaving the word as is.
- Another example of this behavior can be seen with the word “plugin,” which isn’t officially a French word but which most native speakers will understand and not bother to translate.

In [ ]:
# # Warning: Pipeline type "translation" is no longer supported in transformers v5.
# # You must load the model directly (see below) or downgrade to v4.x with:
# # 'pip install "transformers<5.0.0.'
# from transformers import pipeline
# pipeline_task = f"{task}_{lang1}_to_{lang2}"
# translator = pipeline(pipeline_task, model=model_checkpoint)
# translator(sample["translation"]["en"])


model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
inputs = tokenizer(
    sample["translation"]["en"], return_tensors="pt"
).to(model.device)
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("===============")

In [ ]:
sample = split_datasets["train"][172]
print(sample["translation"]["en"])
print(sample["translation"]["fr"])
print("===============")
inputs = tokenizer(
    sample["translation"]["en"], return_tensors="pt"
).to(model.device)
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("===============")

# Processing the data

- If you forget to indicate that you are tokenizing labels, they will be tokenized by the input tokenizer
    - which in the case of a Marian model is not going to go well at all
- If you are using a T5 model (more specifically, one of the `t5-xxx` checkpoints), the model will expect the text inputs to have a prefix indicating the task at hand
    - such as `translate: English to French:`.

In [ ]:
sample = split_datasets["train"][1]
en_sentence = sample["translation"]["en"]
fr_sentence = sample["translation"]["fr"]
inputs = tokenizer(en_sentence, text_target=fr_sentence)
print(inputs)

In [ ]:
def preprocess_function(examples, tokenizer, max_length):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    model_inputs = tokenizer(
        inputs, text_target=targets, max_length=max_length, truncation=True
    )
    return model_inputs

- We don’t pay attention to the attention mask of the targets, as the model won’t expect it.
- Instead, the labels corresponding to a padding token should be set to `-100` so they are ignored in the loss computation.
- This will be done by our data collator later on since we are applying dynamic padding
- but if you use padding here, you should adapt the preprocessing function to set all labels that correspond to the padding token to `-100`.

In [ ]:
tokenized_datasets = split_datasets.map(
    preprocess_function,
    batched=True,
    fn_kwargs = {"tokenizer":tokenizer, "max_length":max_length},
    remove_columns=split_datasets["train"].column_names,
)

# Fine-tuning the model

- The actual code using the `Trainer` will be the same as before, with just one little change
- we use a [`Seq2SeqTrainer`](https://huggingface.co/transformers/main_classes/trainer.html#seq2seqtrainer) here, which is a subclass of `Trainer`
- that will allow us to properly deal with the evaluation, using the `generate()` method to predict outputs from the inputs.

## Data collation

- We’ll need a data collator to deal with the padding for dynamic batching.
- We can’t just use a `DataCollatorWithPadding`, because that only pads the inputs (input IDs, attention mask, and token type IDs).
- Our labels should also be padded to the maximum length encountered in the labels.
- And, as mentioned previously, the padding value used to pad the labels should be `-100` and not the padding token of the tokenizer, to make sure those padded values are ignored in the loss computation.
- This is all done by a [`DataCollatorForSeq2Seq`](https://huggingface.co/transformers/main_classes/data_collator.html#datacollatorforseq2seq).
- Like the `DataCollatorWithPadding`, it takes the `tokenizer` used to preprocess the inputs, but it also takes the `model`.
    - This is because this data collator will also be responsible for preparing the decoder input IDs, which are shifted versions of the labels with a special token at the beginning.
    - Since this shift is done slightly differently for different architectures, the `DataCollatorForSeq2Seq` needs to know the `model` object.

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
for k,v in batch.items():
    print(f"{k} | {v.shape}========")
    print(v)

In [ ]:
print(f"special tokens: {[x for x in tokenizer._special_tokens_map.values() if x]}")
print("=================")
print(tokenizer.decode(59513))
print(tokenizer.decode(0))
print("=================")
en_text = sample["translation"]["en"]
print(en_text)
print(tokenizer.decode(tokenizer.encode(en_text)))
print("==============")
fr_text = sample["translation"]["fr"]
print(fr_text)
print(tokenizer.decode(tokenizer.encode(fr_text)))
print("==============")

## Metrics

### Definition

- The traditional metric used for translation is the [BLEU score](https://en.wikipedia.org/wiki/BLEU), introduced in [a 2002 article](https://aclanthology.org/P02-1040.pdf) by Kishore Papineni et al.
- The BLEU score evaluates how close the translations are to their labels.
- It does not measure the intelligibility or grammatical correctness of the model’s generated outputs, but uses statistical rules to ensure that all the words in the generated outputs also appear in the targets.
- In addition, there are rules that penalize
    - repetitions of the same words if they are not also repeated in the targets (to avoid the model outputting sentences like `"the the the the the"`)
    - and output sentences that are shorter than those in the targets (to avoid the model outputting sentences like `"the"`).

- One weakness with BLEU is that it expects the text to already be tokenized, which makes it difficult to compare scores between models that use different tokenizers.
- So instead, the most commonly used metric for benchmarking translation models today is [SacreBLEU](https://github.com/mjpost/sacrebleu), which addresses this weakness (and others) by standardizing the tokenization step.

In [ ]:
metric = evaluate.load("sacrebleu")

- This metric will take texts as inputs and targets. It is designed to accept several acceptable targets, as there are often multiple acceptable translations of the same sentence
- The dataset we’re using only provides one, but it’s not uncommon in NLP to find datasets that give several sentences as labels.
- So, the predictions should be a list of sentences, but the references should be a list of lists of sentences.

### Examples

In [ ]:
predictions = [
    "This plugin lets you translate web pages between several languages automatically."
]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

- This gets a BLEU score of 46.75, which is rather good
    - for reference, the original Transformer model in the [“Attention Is All You Need” paper](https://arxiv.org/pdf/1706.03762.pdf) achieved a BLEU score of 41.8 on a similar translation task between English and French!
- For more information about the individual metrics, like `counts` and `bp`, see the [SacreBLEU repository](https://github.com/mjpost/sacrebleu/blob/078c440168c6adc89ba75fe6d63f0d922d42bcfe/sacrebleu/metrics/bleu.py#L74).)
- The score can go from 0 to 100, and higher is better.

In [ ]:
print(metric.compute(
    predictions = ["This This This This"],
    references=references
    ))
print("====================")
print(metric.compute(
    predictions = ["This plugin"],
    references=references
    ))
print("====================")

### Implementation

- The feature that `Seq2SeqTrainer` adds to its superclass `Trainer` is the ability to use the `generate()` method during evaluation or prediction.
- **During training**, the model will use the `decoder_input_ids` with an attention mask ensuring it does not use the tokens after the token it’s trying to predict, to speed up training.
- **During inference** we won’t be able to use those since we won’t have labels, so it’s a good idea to evaluate our model with the same setup.
- The decoder performs inference by predicting tokens one by one — something that’s implemented behind the scenes in 🤗 Transformers by the `generate()` method.
- The `Seq2SeqTrainer` will let us use that method for evaluation if we set `predict_with_generate=True`.

- To get from the model outputs to texts the metric can use, we will use the `tokenizer.batch_decode()` method.
- We just have to clean up all the `-100`s in the labels (the tokenizer will automatically do the same for the padding token)

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # In case the model returns more than the prediction logits
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100s in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

### mini batch evaluation example

In [ ]:
for i in range(1, 4):
    print(split_datasets["train"][i]["translation"]["fr"])
print("===========")

sample_batch = data_collator(
    [tokenized_datasets["train"][i] for i in range(1, 4)]
)

for k,v in sample_batch.items():
    print(f"{k} | {v.shape}")
    # print(v[:2])
print("===========")
input_ids = sample_batch["input_ids"].to(model.device)
attention_mask = sample_batch["attention_mask"].to(model.device)
preds = model.generate(input_ids=input_ids, attention_mask=attention_mask)
print(f"generation output: {preds.shape}")
print("===========")
score = compute_metrics((preds, sample_batch["labels"]))
print(f"score: {score}")
print("===========")
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
for pred in decoded_preds: print(pred)

## Train Loop

In [ ]:
# ??Seq2SeqTrainingArguments

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir=ckpt_name,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    push_to_hub=push_to_hub,
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    bf16=True,
)

- Apart from the usual hyperparameters (like learning rate, number of epochs, batch size, and some weight decay), here are a few changes compared to what we saw in the previous sections:
    - We don’t set any regular evaluation, as evaluation takes a while; we will just evaluate our model once before training and after.
    - We set `bf16=True`, which speeds up training on modern GPUs.
    - We set `predict_with_generate=True`, as discussed above.
    - We use `push_to_hub=True` to upload the model to the Hub at the end of each epoch.
---
- Note that while the training happens, each time the model is saved (here, every epoch) it is uploaded to the Hub in the background.
- This way, you will be able to to resume your training on another machine if necessary.

In [ ]:
# ??Seq2SeqTrainer

In [ ]:
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

- Before training, we’ll first look at the score our model gets, to double-check that we’re not making things worse with our fine-tuning.
- This command will take a bit of time

In [ ]:
trainer.evaluate(max_length=max_length)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate(max_length=max_length)

# Push to hub

- Usually, there is no need to say anything as it can infer the right widget from the model class
- but in this case, the same model class can be used for all kinds of sequence-to-sequence problems
- so we specify it’s a translation model.

In [ ]:
trainer.push_to_hub(tags="translation", commit_message="Training complete")

# Inference with the fine-tuned model

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(f"{hf_user_name}/{ckpt_name}")
text = "Default to expanded threads"
print(text)
print("===============")
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("===============")

In [ ]:
text = "Unable to import %1 using the OFX importer plugin. This file is not the correct format."
print(text)
print("===============")
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("===============")